# Statistics & EDA Lab: Understanding Your Data Before Modeling

## Student Practice Notebook

**Name:** __________D RAKESH____________  
**Register Number:** _______AP24110011170_______________  
**Date:** _______28-08-2026____________  

## Learning Objectives

After completing this lab, you will be able to:

- Compute and interpret descriptive statistics beyond `.describe()` (variance, std, skewness)
- Visualize and identify distribution shapes (normal vs skewed)
- Compute and interpret correlation between features
- Run a basic hypothesis test (t-test) and interpret a p-value
- Understand confidence intervals conceptually
- Apply statistical thinking to image and text data

### Why this week matters
You now have clean, scaled, encoded data. Before building any model, a good data scientist asks: **what does this data actually look like? Which features seem related? Is a difference I'm seeing real, or just noise?** That's what statistics answers.

### How to use this notebook
Same as before: **demo → practice → predict-then-run → reflect.**


In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.precision', 3)

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips['tip_pct'] = tips['tip'] / tips['total_bill']
tips.head()


## Part 1: Descriptive Statistics Beyond `.describe()`

`.describe()` gives mean, std, min/max, quartiles. Two more measures matter a lot for understanding data shape:

- **Variance / Std Dev**: how spread out the data is
- **Skewness**: whether the data leans left (negative skew) or right (positive skew), or is symmetric (skew ≈ 0)

### Demonstration


In [ ]:
print("Mean:", tips['total_bill'].mean())
print("Median:", tips['total_bill'].median())
print("Variance:", tips['total_bill'].var())
print("Std Dev:", tips['total_bill'].std())
print("Skewness:", tips['total_bill'].skew())


**Reading skewness:**
- Skew ≈ 0 → roughly symmetric
- Skew > 0 → right-skewed (long tail of high values; mean > median)
- Skew < 0 → left-skewed (long tail of low values; mean < median)


### Student Practice

Compute mean, median, variance, std dev, and skewness for the `tip_pct` column. Based on the mean vs median relationship, is it left-skewed, right-skewed, or roughly symmetric? Write your conclusion.


In [ ]:
mean = tips['tip_pct'].mean()
median = tips['tip_pct'].median()
variance = tips['tip_pct'].var()
std_dev = tips['tip_pct'].std()
skewness = tips['tip_pct'].skew()

print(f"Mean: {mean:.4f}")
print(f"Median: {median:.4f}")
print(f"Variance: {variance:.6f}")
print(f"Std Dev: {std_dev:.4f}")
print(f"Skewness: {skewness:.4f}")

if mean > median:
    print("Conclusion: tip_pct is right-skewed because the mean is greater than the median.")
elif mean < median:
    print("Conclusion: tip_pct is left-skewed because the mean is less than the median.")
else:
    print("Conclusion: tip_pct is roughly symmetric because the mean and median are very close.")


**Your answer:** Two columns can have the same mean but different standard deviations because the values can be spread out differently around that mean. Skewness can also be different if one column has a longer tail on one side. This matters during preprocessing because a feature with high spread or strong skew may need scaling or an outlier treatment before it is used in a model.


## Part 2: Visualizing Distributions

Numbers alone can be misleading — always look at the shape too.

### Demonstration


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))

axes[0].hist(tips['total_bill'], bins=20, edgecolor='black')
axes[0].set_title("Total Bill Distribution")
axes[0].set_xlabel("Total Bill")

axes[1].boxplot(tips['total_bill'])
axes[1].set_title("Total Bill Boxplot")

plt.tight_layout()
plt.show()


### Student Practice — predict then check

**My prediction:** I expect `tip_pct` to be somewhat right-skewed rather than perfectly bell-shaped. The mean is higher than the median, which suggests that some observations with relatively high tip percentages pull the distribution toward the right. The histogram and boxplot should make this easier to see.


In [ ]:
print("Prediction: I expect the tip_pct distribution to be right-skewed rather than perfectly bell-shaped.")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(tips['tip_pct'], bins=20, edgecolor='black')
axes[0].set_title("Tip Percentage Distribution")
axes[0].set_xlabel("Tip Percentage")
axes[0].set_ylabel("Frequency")

axes[1].boxplot(tips['tip_pct'])
axes[1].set_title("Tip Percentage Boxplot")
axes[1].set_ylabel("Tip Percentage")

plt.tight_layout()
plt.show()
print("Check: the graph should show the longer tail on the higher tip-percentage side, which agrees with the prediction.")


## Part 3: Correlation

Correlation measures how strongly two numeric variables move together, from -1 (perfectly inverse) to +1 (perfectly aligned). 0 means no linear relationship.

### Demonstration


In [ ]:
correlation = tips['total_bill'].corr(tips['tip'])
print(f"Correlation between total_bill and tip: {correlation:.3f}")

plt.figure()
plt.scatter(tips['total_bill'], tips['tip'], alpha=0.6)
plt.title("Total Bill vs Tip")
plt.xlabel("Total Bill")
plt.ylabel("Tip")
plt.show()


In [ ]:
numeric_cols = tips.select_dtypes(include=[np.number])
corr_matrix = numeric_cols.corr()
corr_matrix


In [ ]:
plt.figure()
plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


### Student Practice

1. What is the correlation between `size` (party size) and `total_bill`? Is it weak, moderate, or strong?
2. What is the correlation between `size` and `tip_pct`? Compare this to `size` vs `total_bill` — which relationship is stronger, and does that surprise you?


In [ ]:
corr_bill = tips['size'].corr(tips['total_bill'])
corr_tip_pct = tips['size'].corr(tips['tip_pct'])

print(f"Correlation between size and total_bill: {corr_bill:.3f}")
print(f"Correlation between size and tip_pct: {corr_tip_pct:.3f}")

print("The relationship between size and total_bill is much stronger.")
print("The relationship between size and tip_pct is weaker, so party size is more closely related to the total bill than to the percentage tipped.")
print("This makes sense because a larger party usually produces a larger total bill, but the tip percentage does not necessarily increase with party size.")


**Your answer:** Yes. Two variables can have a strong non-linear relationship and still have a correlation close to 0. Correlation mainly measures a straight-line relationship. For example, if the data follows a U-shaped curve, the variables are clearly related, but the positive and negative parts can cancel each other out and give a correlation near 0.


## Part 4: Hypothesis Testing — Is a Difference Real?

You've seen group averages differ (e.g. average tip_pct by day). But is that difference **statistically significant**, or could it just be random variation? A **t-test** compares two group means and gives a **p-value**: the probability of seeing a difference this large (or larger) if there were truly no difference.

**Rule of thumb:** if p-value < 0.05, we consider the difference statistically significant.

### Demonstration: do smokers tip a different percentage than non-smokers?


In [ ]:
smoker_tips = tips[tips['smoker']=='Yes']['tip_pct']
nonsmoker_tips = tips[tips['smoker']=='No']['tip_pct']

print("Smoker mean tip_pct:", smoker_tips.mean())
print("Non-smoker mean tip_pct:", nonsmoker_tips.mean())

t_stat, p_value = stats.ttest_ind(smoker_tips, nonsmoker_tips)
print(f"\nt-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.3f}")

if p_value < 0.05:
    print("Result: statistically significant difference")
else:
    print("Result: NOT a statistically significant difference (could be random variation)")


### Student Practice — predict then check

**My prediction:** I expect lunch and dinner to have different average tip percentages, but I do not expect the difference to be large enough to automatically mean there is a real difference in the population. I will use the t-test and p-value to decide whether the observed difference is statistically significant.


In [ ]:
lunch_tips = tips[tips['time'] == 'Lunch']['tip_pct']
dinner_tips = tips[tips['time'] == 'Dinner']['tip_pct']

print("Prediction: I do not expect a very large difference between lunch and dinner tip percentages, so I predict the result may not be statistically significant.")

t_stat, p_value = stats.ttest_ind(lunch_tips, dinner_tips)
print(f"Lunch mean tip_pct: {lunch_tips.mean():.4f}")
print(f"Dinner mean tip_pct: {dinner_tips.mean():.4f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.3f}")

if p_value < 0.05:
    print("Result: statistically significant difference")
else:
    print("Result: NOT a statistically significant difference")


**Your answer:** It is important to check statistical significance because a difference between two means can happen just because of random variation in the sample. A statistical test helps us decide whether the observed difference is large enough to be considered meaningful rather than immediately assuming that the difference represents a real effect.


## Part 5: Confidence Intervals (Conceptual Introduction)

A **confidence interval (CI)** gives a range where we're reasonably confident the true population mean lies, based on our sample. A 95% CI means: if we repeated this sampling process many times, about 95% of such intervals would contain the true mean.

### Demonstration


In [ ]:
mean_bill = tips['total_bill'].mean()
sem = stats.sem(tips['total_bill'])
ci = stats.t.interval(0.95, len(tips)-1, loc=mean_bill, scale=sem)

print(f"Sample mean total_bill: {mean_bill:.2f}")
print(f"95% Confidence Interval: ({ci[0]:.2f}, {ci[1]:.2f})")


### Student Practice

Compute the 95% confidence interval for `tip_pct`. In one sentence, explain what this interval tells you (in plain language, not just formula terms).


In [ ]:
mean_tip_pct = tips['tip_pct'].mean()
sem_tip_pct = stats.sem(tips['tip_pct'])
ci_tip_pct = stats.t.interval(0.95, len(tips['tip_pct']) - 1, loc=mean_tip_pct, scale=sem_tip_pct)

print(f"Sample mean tip_pct: {mean_tip_pct:.4f}")
print(f"95% Confidence Interval: ({ci_tip_pct[0]:.4f}, {ci_tip_pct[1]:.4f})")
print("In plain language: this interval gives a reasonable range for the true average tip percentage based on the sample of restaurant bills.")


# Mini Project A: Statistical Comparison of Images (Image Track)

Using the image metadata DataFrame you built in Week 2 (filename, brightness, RGB averages, etc.), apply statistical thinking to compare your images.

## Step 1: Rebuild your image metadata table


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from PIL import Image
import numpy as np

filenames = list(uploaded.keys())
rows = []
for name in filenames:
    arr = np.array(Image.open(name))
    rows.append({
        'filename': name,
        'avg_brightness': np.mean(arr),
        'avg_red': np.mean(arr[:,:,0]) if arr.ndim==3 else np.mean(arr),
        'avg_green': np.mean(arr[:,:,1]) if arr.ndim==3 else np.mean(arr),
        'avg_blue': np.mean(arr[:,:,2]) if arr.ndim==3 else np.mean(arr),
        'height': arr.shape[0],
        'width': arr.shape[1],
    })

image_df = pd.DataFrame(rows)
image_df


## Step 2: Statistical analysis

1. Compute the mean, std dev, and skewness of `avg_brightness` across your images.
2. Compute the correlation between `avg_brightness` and `avg_red` (do brighter images tend to have more red?).
3. **If you have at least 2 images you can meaningfully group** (e.g. indoor vs outdoor, or any 2 categories you choose), run a t-test comparing their `avg_brightness`. If you only have very few images, skip the t-test and instead just discuss whether a t-test would even be reliable with so few samples.


In [ ]:
image_statistics = image_df[['avg_brightness', 'avg_red', 'avg_green', 'avg_blue']].agg(['mean', 'std', 'skew']).T
print(image_statistics)

brightness_red_correlation = image_df['avg_brightness'].corr(image_df['avg_red'])
print(f"Correlation between avg_brightness and avg_red: {brightness_red_correlation:.3f}")

print("The correlation shows whether brighter images also tend to have higher average red intensity. A value close to 1 means a strong positive relationship, a value close to -1 means a strong negative relationship, and a value close to 0 means little linear relationship.")

if len(image_df) >= 10:
    brightness_groups = pd.qcut(image_df['avg_brightness'], q=2, labels=['Lower brightness', 'Higher brightness'])
    lower_brightness = image_df.loc[brightness_groups == 'Lower brightness', 'avg_brightness']
    higher_brightness = image_df.loc[brightness_groups == 'Higher brightness', 'avg_brightness']
    t_statistic, p_value = stats.ttest_ind(lower_brightness, higher_brightness)
    print(f"t-statistic: {t_statistic:.3f}")
    print(f"p-value: {p_value:.3f}")
else:
    print("A t-test is skipped because there are too few images for a reliable comparison.")


**Your answer:** No, I would not trust a statistical significance result from only 3–5 images in the same way as a result from 200 or more observations. A very small sample can give an unstable mean, standard deviation, and p-value, and one unusual image can have a large effect on the result. With a larger dataset, the estimates are generally more stable and the statistical test has more information to work with. For a small image set, I would treat the results as exploratory rather than strong evidence.


# Mini Project B: Word Frequency Distribution Analysis (NLP Track)

## Step 1: Build a word frequency table for a longer text

### Demonstration: Zipf's Law preview

A famous finding in linguistics: in most natural language text, word frequency is **highly right-skewed** — a few words (like "the", "a") appear extremely often, and most words appear only once or twice. This is called **Zipf's Law**.


In [ ]:
paragraph = (
    "Data science combines statistics and programming. Statistics helps us understand data. "
    "Programming helps us process data. Machine learning is a part of data science. "
    "Data science uses statistics, programming, and machine learning together. "
    "Understanding statistics is essential for data science."
)

tokens = paragraph.lower().replace('.', '').replace(',', '').split()
tokens_array = np.array(tokens)
vocabulary = np.unique(tokens_array)
counts = [np.sum(tokens_array == w) for w in vocabulary]

word_freq = pd.DataFrame({'word': vocabulary, 'count': counts}).sort_values('count', ascending=False).reset_index(drop=True)
print(word_freq)
print("\nSkewness of word counts:", word_freq['count'].skew())


In [ ]:
plt.figure()
plt.bar(word_freq['word'], word_freq['count'])
plt.xticks(rotation=90)
plt.title("Word Frequency Distribution")
plt.ylabel("Count")
plt.show()


### Student Practice

1. Pick your own longer paragraph (5+ sentences) and build the same word frequency table.
2. Compute the skewness of the word counts. Is it right-skewed (consistent with Zipf's Law)?
3. What percentage of words in your vocabulary appear only once? (This is usually a large percentage — that's Zipf's Law in action.)


In [ ]:
paragraph_text = (
    "Data science is useful because it helps us understand information and make better decisions. "
    "Statistics gives us methods for describing data and finding patterns in observations. "
    "Programming allows us to clean data, analyze it, and build useful applications. "
    "Machine learning uses data to learn patterns that can be used for prediction and classification. "
    "Good data analysis requires us to check distributions, relationships, unusual values, and the quality of the data. "
    "When statistics and programming are used together, large datasets can be studied in a systematic way. "
    "Understanding the data before modeling can help us choose better features and avoid misleading conclusions. "
)

clean_words = re.sub(r'[^a-zA-Z\\s]', '', paragraph_text.lower()).split()
word_array = np.array(clean_words)
unique_words = np.unique(word_array)
word_counts = [np.sum(word_array == current_word) for current_word in unique_words]

student_word_frequency = pd.DataFrame({'word': unique_words, 'count': word_counts}).sort_values('count', ascending=False).reset_index(drop=True)
print(student_word_frequency)

word_count_skewness = student_word_frequency['count'].skew()
print(f"Skewness of word counts: {word_count_skewness:.3f}")

single_occurrence_words = (student_word_frequency['count'] == 1).sum()
unique_word_total = len(student_word_frequency)
single_occurrence_percentage = single_occurrence_words / unique_word_total * 100
print(f"Percentage of vocabulary appearing only once: {single_occurrence_percentage:.2f}%")

if word_count_skewness > 0:
    print("The word-count distribution is right-skewed, which is consistent with the usual pattern described by Zipf's Law.")
else:
    print("The word-count distribution is not clearly right-skewed in this sample.")


## Step 2: Correlation between sentence length and vocabulary diversity

### Demonstration


In [ ]:
sentences = [
    "I like data science.",
    "Machine learning models require lots of clean labeled training data to perform well.",
    "Statistics matters.",
    "Understanding both statistics and programming together is essential for real data science work in industry.",
    "NumPy and pandas are useful tools for data analysis in Python."
]

sentence_stats = []
for s in sentences:
    toks = s.lower().replace('.', '').split()
    sentence_stats.append({
        'sentence': s,
        'length': len(toks),
        'unique_words': len(set(toks))
    })

sentence_df = pd.DataFrame(sentence_stats)
sentence_df['diversity_ratio'] = sentence_df['unique_words'] / sentence_df['length']
sentence_df


In [ ]:
correlation = sentence_df['length'].corr(sentence_df['unique_words'])
print(f"Correlation between sentence length and unique word count: {correlation:.3f}")


### Student Practice

Write 5 sentences of your own, varying in length. Build the same table and compute the correlation between `length` and `unique_words`. Is the correlation strong? Does this match your intuition (longer sentences tend to have more unique words)?


In [ ]:
student_sentences = [
    "I enjoy learning data science concepts.",
    "Python makes data analysis easier when the dataset is organized properly.",
    "Statistics helps us understand patterns, variation, and relationships in real data.",
    "Machine learning models can learn useful patterns from carefully prepared training data and meaningful features.",
    "A good data scientist checks the quality of the dataset before choosing a model because poor data can produce unreliable results."
]

student_sentence_statistics = []
for current_sentence in student_sentences:
    current_tokens = re.sub(r'[^a-zA-Z\\s]', '', current_sentence.lower()).split()
    student_sentence_statistics.append({
        'sentence': current_sentence,
        'length': len(current_tokens),
        'unique_words': len(set(current_tokens))
    })

student_sentence_df = pd.DataFrame(student_sentence_statistics)
student_sentence_df['diversity_ratio'] = student_sentence_df['unique_words'] / student_sentence_df['length']
print(student_sentence_df)

sentence_length_correlation = student_sentence_df['length'].corr(student_sentence_df['unique_words'])
print(f"Correlation between sentence length and unique word count: {sentence_length_correlation:.3f}")

if abs(sentence_length_correlation) >= 0.7:
    print("The correlation is strong. In these sentences, longer sentences generally contain more unique words.")
elif abs(sentence_length_correlation) >= 0.4:
    print("The correlation is moderate. Sentence length and unique word count show a noticeable relationship.")
else:
    print("The correlation is weak. Sentence length and unique word count do not have a strong linear relationship in this sample.")


**Your answer:** `diversity_ratio` is more useful when sentences have very different lengths because raw `unique_words` naturally tends to increase when a sentence becomes longer. The ratio accounts for sentence length by showing what fraction of the words are unique. This makes comparisons between short and long sentences fairer. A short sentence can therefore be compared with a long sentence without assuming that the longer sentence is automatically more diverse just because it contains more words.


# Final Self-Check

- [x] I computed and correctly interpreted skewness (not just mean/std)
- [x] I visualized a distribution and connected the shape to the skewness number
- [x] I computed correlation and correctly judged strength (weak/moderate/strong)
- [x] I ran a t-test and correctly interpreted the p-value (significant or not)
- [x] I computed a confidence interval and can explain what it means in plain language
- [x] I completed Mini Project A (image stats) or B (text distribution/correlation)
- [x] I answered all reflection questions in my own words

## Final Reflection Questions

### 1. Why is it important to look at data visually?

Visualizations such as histograms and boxplots show the actual shape and spread of the data. Two datasets can have similar mean and standard deviation but have very different distributions, outliers, or skewness. Looking at the graphs helps us notice these patterns before making decisions about the data.

### 2. If a t-test gives p = 0.20, what should you conclude?

A p-value of 0.20 is greater than the usual 0.05 threshold, so I would not call the observed difference statistically significant. It means the sample does not provide strong enough evidence to reject the null hypothesis. It does not mean that there is definitely no difference. A real difference could still exist, but this sample does not give enough evidence to establish it.

### 3. How does understanding statistics help with ML modeling?

Understanding distributions helps us identify skewed features, unusual values, and possible outliers. Correlation can show which variables have useful relationships, although correlation alone does not prove causation. This information helps with feature selection, preprocessing, and choosing suitable modeling approaches. It also reduces the chance of building a model without understanding the data first.

### 4. What's one statistics concept you still feel unsure about?

I still want to understand confidence intervals more deeply, especially why a 95% confidence interval should not be interpreted simply as a 95% probability that the fixed population mean is inside one particular interval. I understand the practical idea, but I would like more practice with repeated samples and the interpretation of confidence levels.
